# 🍺 Predicción de Ventas de Cerveza con SARIMA

Este notebook te guía paso a paso para construir un modelo SARIMA de predicción de ventas.

**Flujo del notebook:**
1. Instalación y librerías
2. Carga y exploración de datos
3. Análisis de estacionariedad
4. Identificación de parámetros (ACF / PACF)
5. Entrenamiento del modelo SARIMA
6. Diagnóstico del modelo
7. Predicción y visualización
8. Evaluación de métricas
9. (Bonus) Auto-selección de parámetros con `auto_arima`

## 1. Instalación y librerías

In [ ]:
# Instalar pmdarima para auto_arima (statsmodels ya viene en Colab)
!pip install pmdarima -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('✅ Librerías cargadas correctamente')

## 2. Carga de datos

> **Opción A:** Usamos el dataset clásico de producción mensual de cerveza en Australia (1956–1995).  
> **Opción B:** Sube tu propio archivo CSV con columnas `fecha` y `ventas`.

In [ ]:
# ── OPCIÓN A: Dataset de ejemplo ──────────────────────────────────────────────
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-beer-production-in-austr.csv'
df = pd.read_csv(url, header=0, index_col=0, parse_dates=True)
df.columns = ['ventas']
df.index.name = 'fecha'
df.index = pd.to_datetime(df.index)
df = df.dropna()

print(f'📦 Dataset cargado: {len(df)} registros mensuales')
print(f'📅 Período: {df.index[0].strftime("%b %Y")} → {df.index[-1].strftime("%b %Y")}')
print(f'📊 Rango de ventas: {df.ventas.min():.0f} – {df.ventas.max():.0f}')
df.tail()

In [ ]:
# ── OPCIÓN B: Sube tu propio CSV ───────────────────────────────────────────────
# Descomenta las siguientes líneas si tienes tus propios datos

# from google.colab import files
# uploaded = files.upload()  # Selecciona tu archivo CSV
# import io
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(io.BytesIO(uploaded[filename]), parse_dates=['fecha'], index_col='fecha')
# df.columns = ['ventas']
# print('✅ Tu archivo fue cargado correctamente')
# df.tail()

## 3. Exploración visual de la serie

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Exploración de la serie de ventas de cerveza', fontsize=14, fontweight='bold', y=1.01)

# Serie completa
axes[0,0].plot(df.index, df.ventas, color='#5B4FCF', linewidth=1.2)
axes[0,0].set_title('Serie temporal completa')
axes[0,0].set_ylabel('Ventas (megalitros)')

# Descomposición estacional
decomp = seasonal_decompose(df.ventas, model='additive', period=12)
axes[0,1].plot(decomp.trend.dropna(), color='#E8593C', linewidth=1.5)
axes[0,1].set_title('Tendencia')

axes[1,0].plot(decomp.seasonal[:48], color='#1D9E75', linewidth=1.2)
axes[1,0].set_title('Estacionalidad (primeros 4 años)')

axes[1,1].plot(decomp.resid.dropna(), color='#888780', linewidth=0.8, alpha=0.7)
axes[1,1].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[1,1].set_title('Residuos')

plt.tight_layout()
plt.show()
print('💡 Observa el patrón estacional: la estacionalidad se repite cada 12 meses')

In [ ]:
# Boxplot por mes para ver estacionalidad
df['mes'] = df.index.month
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(14, 5))
df.boxplot(column='ventas', by='mes', ax=ax,
           boxprops=dict(color='#5B4FCF'),
           medianprops=dict(color='#E8593C', linewidth=2),
           whiskerprops=dict(color='#5B4FCF'),
           capprops=dict(color='#5B4FCF'))
ax.set_xticklabels(meses)
ax.set_title('Distribución de ventas por mes')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas (megalitros)')
plt.suptitle('')
plt.tight_layout()
plt.show()
df.drop(columns=['mes'], inplace=True)

## 4. Prueba de estacionariedad (ADF)

SARIMA requiere que la serie sea **estacionaria** (media y varianza constantes). Usamos la prueba Augmented Dickey-Fuller.

- Si **p-value < 0.05** → la serie ES estacionaria → `d=0`
- Si **p-value ≥ 0.05** → aplicar diferenciación (`d=1` o `d=2`)

In [ ]:
def test_estacionariedad(serie, nombre='Serie'):
    result = adfuller(serie.dropna())
    print(f'\n📊 Prueba ADF — {nombre}')
    print(f'   Estadístico ADF : {result[0]:.4f}')
    print(f'   p-value         : {result[1]:.4f}')
    print(f'   Valores críticos: {{\'1%\': {result[4]["1%"]:.2f}, \'5%\': {result[4]["5%"]:.2f}, \'10%\': {result[4]["10%"]:.2f}}}')
    if result[1] < 0.05:
        print('   ✅ Serie ESTACIONARIA (rechaza H0 de raíz unitaria)')
    else:
        print('   ⚠️  Serie NO estacionaria → aplicar diferenciación')
    return result[1] < 0.05

es_estacionaria = test_estacionariedad(df.ventas, 'Ventas originales')

if not es_estacionaria:
    diff1 = df.ventas.diff().dropna()
    test_estacionariedad(diff1, 'Ventas diferenciadas (d=1)')

## 5. Identificación de parámetros: ACF y PACF

- **ACF (Autocorrelation Function)** → ayuda a elegir `q` y `Q`
- **PACF (Partial ACF)** → ayuda a elegir `p` y `P`
- Los picos en los rezagos múltiplos de 12 indican el componente estacional

In [ ]:
# Diferenciamos la serie para análisis (d=1, D=1, m=12)
serie_diff = df.ventas.diff(12).diff(1).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(serie_diff, lags=48, ax=axes[0], color='#5B4FCF', zero=False)
axes[0].set_title('ACF — picos estacionales cada 12 rezagos → q, Q')

plot_pacf(serie_diff, lags=48, ax=axes[1], color='#E8593C', method='ywm', zero=False)
axes[1].set_title('PACF — corte en rezago inicial → p, P')

for ax in axes:
    ax.axvline(12, color='#1D9E75', linewidth=1, linestyle='--', alpha=0.5, label='Lag 12')
    ax.axvline(24, color='#1D9E75', linewidth=1, linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()
plt.show()
print('💡 Interpreta: picos significativos (fuera de la banda azul) definen los parámetros')

## 6. División train / test y entrenamiento del modelo

In [ ]:
# ── Parámetros SARIMA ──────────────────────────────────────────────────────────
# Ajusta estos valores según lo que observaste en ACF/PACF o usa auto_arima (celda 9)
p, d, q = 1, 1, 1          # Parte no estacional
P, D, Q, m = 1, 1, 1, 12  # Parte estacional (m=12 para datos mensuales)

# ── División de datos (80% train / 20% test) ──────────────────────────────────
n_test = 24  # Últimos 24 meses como test
train = df.ventas[:-n_test]
test  = df.ventas[-n_test:]

print(f'📚 Train: {len(train)} observaciones ({train.index[0].strftime("%b %Y")} – {train.index[-1].strftime("%b %Y")})')
print(f'🧪 Test:  {len(test)} observaciones ({test.index[0].strftime("%b %Y")} – {test.index[-1].strftime("%b %Y")})')

# ── Entrenamiento ─────────────────────────────────────────────────────────────
print(f'\n⏳ Entrenando SARIMA({p},{d},{q})({P},{D},{Q},{m})...')
modelo = SARIMAX(train,
                 order=(p, d, q),
                 seasonal_order=(P, D, Q, m),
                 enforce_stationarity=False,
                 enforce_invertibility=False)
resultado = modelo.fit(disp=False)
print('✅ Modelo entrenado!')
print(resultado.summary())

## 7. Diagnóstico del modelo

Un buen modelo SARIMA debe tener residuos que se comporten como **ruido blanco** (aleatorios, sin estructura).

In [ ]:
resultado.plot_diagnostics(figsize=(16, 9))
plt.suptitle('Diagnóstico de residuos del modelo SARIMA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📋 Qué buscar en el diagnóstico:')
print('  • Residuos estandarizados → sin patrón visible, centrados en 0')
print('  • Histograma → distribución normal (campana de Gauss)')
print('  • Q-Q Plot → puntos alineados sobre la diagonal roja')
print('  • Correlograma → todas las barras dentro de la banda azul')

## 8. Predicción y visualización

In [ ]:
# Predicciones sobre el conjunto de test
pred_test = resultado.get_prediction(start=test.index[0], end=test.index[-1])
pred_ci   = pred_test.conf_int(alpha=0.05)  # Intervalo de confianza 95%
pred_mean = pred_test.predicted_mean

fig, ax = plt.subplots(figsize=(16, 6))

# Datos históricos
ax.plot(train.index, train, label='Datos de entrenamiento', color='#5B4FCF', linewidth=1.2)
ax.plot(test.index,  test,  label='Datos reales (test)',    color='#1D9E75', linewidth=2)

# Predicción
ax.plot(pred_mean.index, pred_mean, label='Predicción SARIMA', color='#E8593C', linewidth=2, linestyle='--')

# Intervalo de confianza
ax.fill_between(pred_ci.index,
                pred_ci.iloc[:, 0],
                pred_ci.iloc[:, 1],
                color='#E8593C', alpha=0.15, label='Intervalo de confianza 95%')

ax.axvline(test.index[0], color='gray', linestyle=':', linewidth=1, label='Inicio del test')
ax.set_title('Predicción de ventas de cerveza — SARIMA', fontsize=14)
ax.set_ylabel('Ventas (megalitros)')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 9. Métricas de evaluación

In [ ]:
mae  = mean_absolute_error(test, pred_mean)
rmse = np.sqrt(mean_squared_error(test, pred_mean))
mape = np.mean(np.abs((test - pred_mean) / test)) * 100

print('📏 Métricas del modelo sobre datos de test:')
print(f'   MAE  (Error Absoluto Medio)        : {mae:.2f} megalitros')
print(f'   RMSE (Raíz del Error Cuadrático)   : {rmse:.2f} megalitros')
print(f'   MAPE (Error Porcentual Absoluto)    : {mape:.2f}%')
print(f'   AIC  (criterio de información)     : {resultado.aic:.2f}')
print(f'   BIC  (criterio bayesiano)           : {resultado.bic:.2f}')
print()
print('💡 Interpretación del MAPE:')
print('   < 10% → Excelente  |  10-20% → Bueno  |  20-50% → Aceptable  |  > 50% → Mejorar modelo')

## 10. Predicción futura (horizonte personalizable)

In [ ]:
# ── Ajusta cuántos meses quieres predecir ──────────────────────────────────────
MESES_FUTURO = 24

# Re-entrenamos con todos los datos
modelo_final = SARIMAX(df.ventas,
                       order=(p, d, q),
                       seasonal_order=(P, D, Q, m),
                       enforce_stationarity=False,
                       enforce_invertibility=False)
resultado_final = modelo_final.fit(disp=False)

forecast = resultado_final.get_forecast(steps=MESES_FUTURO)
forecast_mean = forecast.predicted_mean
forecast_ci   = forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df.index[-60:], df.ventas[-60:], label='Histórico (últimos 5 años)', color='#5B4FCF', linewidth=1.5)
ax.plot(forecast_mean.index, forecast_mean, label=f'Predicción ({MESES_FUTURO} meses)', color='#E8593C', linewidth=2)
ax.fill_between(forecast_ci.index,
                forecast_ci.iloc[:, 0],
                forecast_ci.iloc[:, 1],
                color='#E8593C', alpha=0.15, label='Intervalo de confianza 95%')

ax.axvline(df.index[-1], color='gray', linestyle=':', linewidth=1, label='Último dato real')
ax.set_title(f'Predicción de ventas — próximos {MESES_FUTURO} meses', fontsize=14)
ax.set_ylabel('Ventas (megalitros)')
ax.legend()
plt.tight_layout()
plt.show()

print('\n📅 Predicciones por mes:')
forecast_df = pd.DataFrame({
    'Predicción': forecast_mean.round(2),
    'Límite inferior (95%)': forecast_ci.iloc[:,0].round(2),
    'Límite superior (95%)': forecast_ci.iloc[:,1].round(2)
})
forecast_df.index = forecast_df.index.strftime('%b %Y')
print(forecast_df.to_string())

## 11. (Bonus) Auto-selección de parámetros con `auto_arima`

> ¿No sabes qué valores usar para `p, d, q, P, D, Q`? `auto_arima` prueba combinaciones automáticamente y selecciona la mejor según el criterio AIC.

In [ ]:
print('🔍 Buscando los mejores parámetros automáticamente...')
print('   (Esto puede tardar 1-3 minutos)')

auto_model = auto_arima(
    train,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    m=12,                    # Estacionalidad mensual
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    d=None,                  # Se determina automáticamente
    D=None,                  # Se determina automáticamente
    seasonal=True,
    information_criterion='aic',
    stepwise=True,           # Búsqueda eficiente
    suppress_warnings=True,
    error_action='ignore',
    n_fits=50
)

print(f'\n✅ Mejor modelo encontrado: SARIMA{auto_model.order}{auto_model.seasonal_order}')
print(f'   AIC: {auto_model.aic():.2f}')
print(auto_model.summary())

In [ ]:
# Comparar auto_arima vs modelo manual
pred_auto = auto_model.predict(n_periods=n_test)
pred_auto_series = pd.Series(pred_auto, index=test.index)

mae_auto  = mean_absolute_error(test, pred_auto_series)
mape_auto = np.mean(np.abs((test - pred_auto_series) / test)) * 100

print('📊 Comparativa de modelos en datos de test:')
print(f'   Modelo manual  → MAE: {mae:.2f} | MAPE: {mape:.2f}%')
print(f'   Auto ARIMA     → MAE: {mae_auto:.2f} | MAPE: {mape_auto:.2f}%')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test, label='Real', color='#1D9E75', linewidth=2)
ax.plot(test.index, pred_mean, label=f'Manual SARIMA ({mape:.1f}% MAPE)', color='#E8593C', linestyle='--', linewidth=1.8)
ax.plot(test.index, pred_auto_series, label=f'Auto ARIMA ({mape_auto:.1f}% MAPE)', color='#5B4FCF', linestyle=':', linewidth=1.8)
ax.set_title('Comparativa: modelo manual vs auto_arima')
ax.legend()
plt.tight_layout()
plt.show()

---
## 📌 Resumen de parámetros SARIMA

| Parámetro | Qué controla | Cómo determinarlo |
|-----------|-------------|-------------------|
| `p` | Rezagos autorregresivos | Corte en PACF |
| `d` | Diferenciaciones | Prueba ADF (0 si estacionaria) |
| `q` | Términos de media móvil | Corte en ACF |
| `P` | AR estacional | Pico en PACF en lag múltiplo de m |
| `D` | Diferenciación estacional | Prueba ADF sobre serie diferenciada estacionalm. |
| `Q` | MA estacional | Pico en ACF en lag múltiplo de m |
| `m` | Periodo estacional | Conocimiento del dominio (12=mensual, 4=trimestral) |

## 🚀 Próximos pasos sugeridos

1. **SARIMAX** — agrega variables externas (temperatura, precio, festividades)
2. **Prophet** — pruébalo como alternativa y compara MAPE
3. **Cross-validation temporal** — usa `TimeSeriesSplit` para una evaluación más robusta
4. **Ensemble** — combina predicciones de varios modelos para reducir el error